This page is dedicated to utilize BlackJAX package to reproduce the Numerical Experiment from Section 4 of Margossian et al. and run independent tests for improved workflow, together with Arviz for better nested R-hat statistics calculation.

**To make it run on GPUs**

In [1]:
# Remove the stable TFP package
!pip uninstall -y tensorflow-probability
!pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt
# Install nightly TFP for JAX
!pip install -Uq tfp-nightly[jax]
# Install Inference Gym
!pip install inference-gym
# need to keep it cuda13 cuz blackjax drop cuda12 support
!pip install -Uq "jax[cuda13]" blackjax optax arviz-base arviz-stats

Found existing installation: tensorflow-probability 0.25.0
Uninstalling tensorflow-probability-0.25.0:
  Successfully uninstalled tensorflow-probability-0.25.0
Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: jax-cuda12-plugin 0.7.2
Uninstalling jax-cuda12-plugin-0.7.2:
  Successfully uninstalled jax-cuda12-plugin-0.7.2
Found existing installation: jax-cuda12-pjrt 0.7.2
Uninstalling jax-cuda12-pjrt-0.7.2:
  Successfully uninstalled jax-cuda12-pjrt-0.7.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 95.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not 

In [1]:
# run those checks if package compatibility is in trouble

# import jax
# import tensorflow_probability as tfp
# import jaxlib

# print("jax:", jax.__version__)
# print("jaxlib:", jaxlib.__version__)
# print("TFP:", tfp.__version__)

# !pip show jax
# !pip show jaxlib
# !pip show blackjax

# import jax.numpy as jnp

# import blackjax

# import tensorflow_probability.substrates.jax as tfp
# import inference_gym.using_jax as gym

# print("JAX:", jax.__version__)
# print("BlackJAX:", blackjax.__version__)
# print("TFP:", tfp.__version__)
# print("Inference Gym imported successfully!")

jax: 0.11.0
jaxlib: 0.11.0
TFP: 0.26.0-dev20260807
Name: jax
Version: 0.11.0
Summary: Differentiate, compile, and transform Numpy code.
Home-page: https://github.com/jax-ml/jax
Author: JAX team
Author-email: jax-dev@google.com
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: jaxlib, ml_dtypes, numpy, opt_einsum, scipy
Required-by: blackjax, dopamine_rl, flax, optax, orbax-checkpoint
Name: jaxlib
Version: 0.11.0
Summary: XLA library for JAX
Home-page: https://github.com/jax-ml/jax
Author: JAX team
Author-email: jax-dev@google.com
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: ml_dtypes, numpy, scipy
Required-by: blackjax, dopamine_rl, jax, optax
Name: blackjax
Version: 1.6.2
Summary: Flexible and fast sampling in Python
Home-page: https://github.com/blackjax-devs/blackjax
Author: 
Author-email: The Blackjax team <remi@thetypicalset.com>
License: Apache License 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: 

In [32]:
print("ArviZ:", avs.__version__)

ArviZ: 1.2.0


**Necessary Packages and Other Settings**

In [1]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [2]:
import numpy as np

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
import inference_gym.using_jax as gym
import jaxlib

import blackjax

import optax

import arviz as az
import arviz_stats as avs

import matplotlib.pyplot as plt

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

import warnings
warnings.filterwarnings('ignore')

import psutil

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

[CudaDevice(id=0)]
gpu


In [3]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [4]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

**Rosenbrock Example**

In [5]:
target = gym.targets.VectorModel(
    gym.targets.Banana(),
    flatten_sample_transformations=True
)

num_dimensions = target.event_shape[0]

print("Target:", type(target))
print("Dimensions:", num_dimensions)
print("Event shape:", target.event_shape)

Target: <class 'inference_gym.dynamic.backend_jax.targets.vector_model.VectorModel'>
Dimensions: 2
Event shape: (2,)


In [6]:
def logdensity(x):
    y = target.default_event_space_bijector(x)
    fldj = target.default_event_space_bijector.forward_log_det_jacobian(x)
    return target.unnormalized_log_prob(y) + fldj

# x = jnp.zeros(num_dimensions)

# print("x:", x)
# print("logdensity(x):", logdensity(x))

In [7]:
num_chains = num_chains_short      # 2048
num_super_chains = num_super_chains # 16
num_sub_chains = num_chains // num_super_chains  # 128

offset = 2.0

def initialize(key, n):
    return (
        10 * random.normal(key, (n, num_dimensions))
        + offset
    )

In [8]:
key = random.PRNGKey(0)

key, init_key = random.split(key)

initial_position_super = initialize(
    init_key,
    num_super_chains
)

initial_position = jnp.repeat(
    initial_position_super,
    num_sub_chains,
    axis=0
)

# print("Super-chain positions:", initial_position_super.shape)
# print("Chain positions:", initial_position.shape)
# key

In [9]:
num_warmup_test = 10
initial_step_size = 1.

In [10]:
warmup = blackjax.chees_adaptation(
    logdensity,
    num_chains=num_chains,
    target_acceptance_rate=0.75,
)
optimizer = optax.adam(learning_rate=0.01)

In [11]:
# 7s for W = 10
key_warmup, key_sample = random.split(key)

(last_states, parameters), _= warmup.run(
    key_warmup,
    initial_position,
    initial_step_size,
    optimizer,
    num_warmup_test,
)

In [12]:
sample_keys = random.split(key_sample, num_chains)

kernel = blackjax.dhmc(logdensity, **parameters).step
new_states_1, info = jax.vmap(kernel)(sample_keys, last_states)

In [13]:
new_states_2, info = jax.vmap(kernel)(sample_keys, new_states_1)

In [14]:
new_states_3, info = jax.vmap(kernel)(sample_keys, new_states_2)

In [15]:
new_states_4, info = jax.vmap(kernel)(sample_keys, new_states_3)

In [16]:
# plt.figure(figsize=(8, 6))

# plt.scatter(
#     initial_position[:, 0],
#     initial_position[:, 1],
#     c="red",
#     label="Initial"
# )

# plt.scatter(
#     last_states.position[:, 0],
#     last_states.position[:, 1],
#     c="blue",
#     label="After warmup"
# )

# plt.scatter(
#     new_states.position[:, 0],
#     new_states.position[:, 1],
#     c="green",
#     label="After 1 step"
# )

# plt.scatter(
#     another_new_states.position[:, 0],
#     another_new_states.position[:, 1],
#     c="purple",
#     label="After 2 steps"
# )

# plt.xlabel("$x_1$")
# plt.ylabel("$x_2$")
# plt.legend()
# plt.show()

In [27]:
def _reduce_variance_interval(x, axis=None, biased=True, keepdims=False):
    # ddof=0 is biased variance (N), ddof=1 is unbiased variance (N-1)
    ddof = 0 if biased else 1
    return jnp.var(x, axis=axis, ddof=ddof, keepdims=keepdims)

def nested_rhat_constrained(result_state, num_super_chains,idx):
    # since we use only N=1, W_k is reduced to 0

    num_sub_chains = result_state.shape[0] // num_super_chains
    num_dimensions = result_state.shape[1]

    chain_states = result_state.reshape(1, -1, num_sub_chains, num_dimensions)
    # chain_states.shape = (1,16,128,2)
    # f_bar 1*k is:
    mean_subchain = jnp.mean(chain_states, axis=2)
    # mean_subchain.shape =(1,16,2)

    # f_bar **K is:
    mean_superchain = jnp.mean(mean_subchain, axis=1)
    # mean_superchain.shape = (1,2)

    variance_chain = _reduce_variance_interval(chain_states, axis=2, biased=False)
    # print(variance_chain.shape) # (1,16,2)
    W = jnp.mean(variance_chain, axis=1)
    # print(f"W dim: {W.shape}") # (1,2)
    B = _reduce_variance_interval(mean_subchain, axis=1, biased=False) # variance of between super chain

    r_hat = jnp.sqrt(1+B/W)[:,idx]
    return r_hat

In [28]:
nested_rhat_constrained(new_states_1.position, num_super_chains, 0)

Array([1.3116938], dtype=float32)

In [29]:
nested_rhat_constrained(new_states_1.position, num_super_chains, 1)

Array([1.1711314], dtype=float32)

In [30]:
nested_rhat_constrained(new_states_4.position, num_super_chains, 1)

Array([1.0388031], dtype=float32)

**Below is the new way**

In [34]:
chain_ids = np.repeat(
    np.arange(0, num_super_chains),
    num_chains_short // num_super_chains
)

In [45]:
# from src/arviz_stats/base/diagnostics.py
def _rhat_nested(ary, superchain_ids):
        ary = np.asarray(ary)
        nchains, niterations = ary.shape

        # Check that all chains are assigned a superchain
        if len(superchain_ids) != nchains:
            raise ValueError("Length of superchain_ids not equal to number of chains")

        # Check that superchains have equal length
        superchain_counts = np.bincount(superchain_ids)
        nchains_per_superchain = np.max(superchain_counts)

        if nchains_per_superchain != np.min(superchain_counts):
            raise ValueError("Number of chains per superchain is not the same for each superchain")

        superchains = np.unique(superchain_ids)

        # Compute chain means and variances
        chain_mean = np.mean(ary, axis=1)
        chain_var = np.var(ary, axis=1, ddof=1)

        # mean of superchains calculated by only including specified chains
        # (equation 4 in Margossian et al. 2024)
        superchain_mean = np.array([np.mean(chain_mean[superchain_ids == k]) for k in superchains])

        # between-chain variance estimate (Bhat_k in equation 7 in Margossian et al. 2024)
        if nchains_per_superchain == 1:
            var_between_chain = np.zeros(len(superchains))
        else:
            var_between_chain = np.array(
                [np.var(chain_mean[superchain_ids == k], ddof=1) for k in superchains]
            )

        #  within-chain variance estimate (What_k in equation 7 in Margossian et al. 2024)
        if niterations == 1:
            var_within_chain = np.zeros(len(superchains))
        else:
            var_within_chain = np.array(
                [np.mean(chain_var[np.where(superchain_ids == k)[0]]) for k in superchains]
            )

        # between-superchain variance (Bhat_nu in equation 6 in Margossian et al. 2024)
        var_between_superchain = np.var(superchain_mean, ddof=1)

        # within-superchain variance (What_nu in equation 7 in Margossian et al. 2024)
        var_within_superchain = np.mean(var_within_chain + var_between_chain)

        # nested Rhat (Rhat_nu in equation 8 in Margossian et al. 2024)
        return np.sqrt(1 + var_between_superchain / var_within_superchain)

In [48]:
test = new_states_1.position[:, 0][:,None]
_rhat_nested(test,chain_ids)

np.float64(1.3116938552394848)

In [40]:
test4 = np.stack(
    [
        np.asarray(new_states_1.position[:, 0]),
        np.asarray(new_states_2.position[:, 0]),
        np.asarray(new_states_3.position[:, 0]),
        np.asarray(new_states_4.position[:, 0]),
    ],
    axis=1,
)

print(test4.shape)

(2048, 4)


In [35]:
x = avs.rhat_nested(test4,superchain_ids=chain_ids)
print(x)
x

1.1801130673412457


array(1.18011307)

In [38]:
x = avs.rhat_nested(
    test4,
    superchain_ids=chain_ids,
    method="rank"
)

print(x)

1.1801130673412457


In [42]:
test4Double = np.stack(
    [
        np.asarray(new_states_1.position),
        np.asarray(new_states_2.position),
        np.asarray(new_states_3.position),
        np.asarray(new_states_4.position),
    ],
    axis=1,
)

print(test4Double.shape)

(2048, 4, 2)


In [43]:
x = avs.rhat_nested(
    test4Double,
    superchain_ids=chain_ids,
)

print(x)

[1.18011307 1.05275971]


The low-level nested-R-hat calculation handles the single-draw case by setting the within-chain variance to zero. However, the current arviz-stats public implementation applies min_draws=4 validation to all nested-R-hat methods, including identity, so single-draw inputs return NaN before reaching the underlying calculation.